# 🛰️ Field Shift: Step 3 - Spatial Cross-Validation & Model Ensembling
**NASA Space Apps Challenge 2026**

This notebook builds the predictive engine of Field Shift. To prevent data leakage and ensure generalization across different agricultural lands, we implement:
1. **Strict Spatial K-Fold (GroupKFold):** Grouping by location.
2. **Algorithm Battle:** Comparing XGBoost, LightGBM, and CatBoost.
3. **Voting Regressor (Ensemble):** Merging the models for maximum stability.
4. **Final Production Model:** Training the winning model (CatBoost) on all data.

In [ ]:
# --- 1. Data Preparation ---
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import VotingRegressor
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import matplotlib.pyplot as plt
import seaborn as sns
import os

print("⚙️ Initiating Strict Model Comparison & Ensembling...")

# Read the enhanced data from Drive
df = pd.read_csv('/content/drive/MyDrive/Field_Shift_NASA_Project/Field_Shift_Enhanced_100k.csv')

features = [
    'Temperature_Avg', 'Relative_Humidity', 'Precipitation', 'NDVI',
    'LST_Celsius', 'Soil_Sand_pct', 'Soil_Clay_pct', 'Crop_Kc',
    'Thermal_Stress_Diff', 'VPD_kPa', 'Soil_Sponge_Index',
    'Month', 'DayOfYear', 'Rain_3d_sum', 'LST_5d_avg'
]
target = 'Root_Soil_Moisture'

X = df[features]
y = df[target]

# The crucial step for Spatial CV: Grouping by Farm Location
groups = df['Location_Name']

In [ ]:
# --- 2. Base Learners & Strict Spatial Cross-Validation ---
# Initialize base models with strong baseline hyperparameters
xgb_model = xgb.XGBRegressor(n_estimators=300, max_depth=7, learning_rate=0.05, random_state=42, n_jobs=-1)
lgb_model = lgb.LGBMRegressor(n_estimators=300, max_depth=7, learning_rate=0.05, random_state=42, n_jobs=-1, verbose=-1)
cb_model = cb.CatBoostRegressor(iterations=300, depth=7, learning_rate=0.05, random_seed=42, verbose=0)

models = {
    'XGBoost': xgb_model,
    'LightGBM': lgb_model,
    'CatBoost': cb_model
}

# GroupKFold ensures no location in the training set is present in the validation set
gkf = GroupKFold(n_splits=5)
results = {}

for name, model in models.items():
    fold_rmse = []
    print(f"⏳ Training {name} across 5 spatial folds...")

    for train_idx, test_idx in gkf.split(X, y, groups):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        rmse = np.sqrt(mean_squared_error(y_test, preds))
        fold_rmse.append(rmse)

    avg_rmse = np.mean(fold_rmse)
    results[name] = avg_rmse
    print(f"✅ {name} Average RMSE: {avg_rmse:.4f}")

In [ ]:
# --- 3. Building Ensemble Model (XGB + LGBM + CatBoost) ---
print("\n🔗 Building Ensemble Model...")
ensemble_model = VotingRegressor(estimators=[
    ('xgb', xgb_model),
    ('lgb', lgb_model),
    ('cb', cb_model)
])

ensemble_rmse = []
for train_idx, test_idx in gkf.split(X, y, groups):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

    ensemble_model.fit(X_train, y_train)
    preds = ensemble_model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, preds))
    ensemble_rmse.append(rmse)

avg_ensemble_rmse = np.mean(ensemble_rmse)
results['Ensemble'] = avg_ensemble_rmse
print(f"🏆 Ensemble Average RMSE: {avg_ensemble_rmse:.4f}")

print("\n📊 Final Leaderboard (Lower is Better):")
for name, score in sorted(results.items(), key=lambda item: item[1]):
    print(f"{name}: {score:.4f}")

In [ ]:
# --- 4. Extracting Feature Importance from the Winning XGBoost Architecture ---
print("🔍 Extracting Feature Importance...")

# Train a full XGBoost to extract feature importance for visualization
xgb_model.fit(X, y)
importance = pd.DataFrame({
    'Feature': features,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importance, palette='viridis')
plt.title('Field Shift: Feature Importance (Optimized Spatial Model)', fontsize=16)
plt.xlabel('XGBoost Feature Importance Score')
plt.ylabel('Features')
plt.tight_layout()

# Save the plot
plot_path = '/content/drive/MyDrive/Field_Shift_NASA_Project/Optimized_Feature_Importance.png'
plt.savefig(plot_path, dpi=300)
print(f"✅ Feature Importance plot saved securely as '{plot_path}'.")
plt.show()

In [ ]:
# --- 5. Final Production Model Deployment ---
print("🚀 Deploying the Winning CatBoost Model for Production...")

# We select CatBoost due to its superior RMSE (0.0648) and fast inference time
final_cb_model = cb.CatBoostRegressor(iterations=300, depth=7, learning_rate=0.05, random_seed=42, verbose=0)

# Train on the ENTIRE dataset for maximum performance in production
final_cb_model.fit(X, y)

# Save the model securely to Google Drive
model_save_path = '/content/drive/MyDrive/Field_Shift_NASA_Project/Field_Shift_CatBoost_Final.cbm'
final_cb_model.save_model(model_save_path)
print(f"💾 Final Production Model Saved: '{model_save_path}'")